# DASHBOARD RISK MANAGEMENT BRI: RESPONSIBLE AI YANG ETIS & TRANSPARAN
## BFLP Hari 7: SHAP, LIME, Bias ML, Prinsip Etika ML (5 Pilar Bank BRI)
### Tugas Terpadu: Langkah 1 s.d. Langkah 4 & Laporan Kepatuhan OJK/BI

## **TUGAS B-1**
Langkah 1: SHAP & LIME, Sumber & Mitigasi Bias ML, Prinsip Etika ML
- 5 Pilar Bank BRI: Fairness, Explainability, Accountability, Transparency, Privacy & Security
- 3 Prinsip OJK Tata Kelola AI Perbankan Indonesia: Keadilan Non-Diskriminatif, Keterjelasan Algoritma, Pengawasan Manusia (Human-in-the-Loop)

## **TUGAS B-2**
Langkah 2: Analisis Bias pada Model Prediksi Gagal Bayar

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
from xgboost import XGBClassifier

pd.set_option('display.max_columns', None)
plt.rcParams['figure.dpi'] = 100
RANDOM_STATE = 42

## 0. Load & Inspeksi Data 10.000 Debitur BRI

In [2]:
try:
    df = pd.read_csv('dummy_credit_risk_BRI_10000_clean_imbalanced.csv')
except Exception:
    df = pd.read_csv('public/dummy_credit_risk_BRI_10000_clean_imbalanced.csv')
print('Shape:', df.shape)
df.head()

## 1. Pelatihan Model: Random Forest vs XGBoost

In [3]:
cat_cols = ['status_pekerjaan', 'kode_pos']
num_cols = ['income', 'dsr', 'ltv', 'dpd', 'durasi_pinjaman', 'usia']
X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, drop_first=False)
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)

xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=RANDOM_STATE)
xgb.fit(X_train, y_train)
print('Model training selesai!')

## 2. Audit Fairness dengan AIF360: Disparate Impact & Mean Difference

In [4]:
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric

def hitung_fairness(sub_df, label_col, privileged_grp, unprivileged_grp, protected_attr='status_pekerjaan'):
    d = sub_df[[label_col]].copy()
    d[protected_attr] = (sub_df[protected_attr] == privileged_grp).astype(int)
    dataset = BinaryLabelDataset(df=d, label_names=[label_col], protected_attribute_names=[protected_attr], favorable_label=0, unfavorable_label=1)
    metric = BinaryLabelDatasetMetric(dataset, privileged_groups=[{protected_attr: 1}], unprivileged_groups=[{protected_attr: 0}])
    return metric.disparate_impact(), metric.mean_difference()

df_pred = df.copy()
df_pred['default_pred'] = xgb.predict(X[X.columns])

hasil = []
for grp in ['Buruh', 'Lainnya', 'Wiraswasta']:
    sub_hist = df[df['status_pekerjaan'].isin(['PNS', grp])]
    sub_pred = df_pred[df_pred['status_pekerjaan'].isin(['PNS', grp])]
    di_h, md_h = hitung_fairness(sub_hist, 'default', 'PNS', grp)
    di_p, md_p = hitung_fairness(sub_pred, 'default_pred', 'PNS', grp)
    hasil.append({'grup': f'PNS vs {grp}', 'DI_hist': di_h, 'DI_pred': di_p, 'MD_hist': md_h, 'MD_pred': md_p})

hasil_df = pd.DataFrame(hasil)
hasil_df

## 3. Simulasi Proxy Discrimination via `kode_pos` & Mitigasi Reweighing

In [5]:
# Injeksi bias lokasi (Jakarta dimaafkan 25%, Lainnya dijatuhkan 25%)
rng = np.random.RandomState(0)
default_injeksi = df['default'].copy()
mask_jkt = (df['kode_pos'] == 'Jakarta') & (default_injeksi == 1)
default_injeksi[mask_jkt & (rng.rand(len(df)) < 0.25)] = 0
mask_lain = (df['kode_pos'] == 'Lainnya') & (default_injeksi == 0)
default_injeksi[mask_lain & (rng.rand(len(df)) < 0.25)] = 1

df_inj = df.copy()
df_inj['default_injeksi'] = default_injeksi
print('Simulasi proxy discrimination selesai!')

## 4. SHAP & LIME Interpretability Report

In [6]:
import shap
from lime.lime_tabular import LimeTabularExplainer

explainer = shap.TreeExplainer(xgb)
X_sample = X_test.sample(min(1000, len(X_test)), random_state=RANDOM_STATE)
shap_values = explainer(X_sample)
print('SHAP explainer siap!')